# ds003876 — Reconstructed iEEG Biomarker Pipeline

Reconstructed from our working session. Includes preprocessing, 10 s windowing, spectral + aperiodic features, parallel coherence, RZ mapping, and preliminary RZ biomarker comparison. HFO, neural fragility, and learned graph methods are deferred.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import mne
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("MNE:", mne.__version__)
data_dir=Path("/workspace/ds003876")
processed_dir=Path("/workspace/ds003876_processed"); processed_dir.mkdir(parents=True,exist_ok=True)


NumPy: 1.24.4
Pandas: 2.2.1
MNE: 1.12.1


In [ ]:
%pip install --force-reinstall --no-cache-dir \
    numpy==1.24.4 \
    pandas==2.2.1 \
    pyarrow==14.0.2 \
    mne==1.12.1 \
    specparam --no-deps

## 1. Find recordings and channel TSVs

In [2]:
edf_files=sorted(data_dir.rglob("*_task-interictal*_ieeg.edf"))
channel_files=sorted(data_dir.rglob("*_channels.tsv"))
print("EDF recordings:",len(edf_files))
print("Channel TSVs:",len(channel_files))


EDF recordings: 54
Channel TSVs: 54


In [3]:
from pathlib import Path
import re

data_dir = Path("/workspace/ds003876")

# ---------------------------------------------------------
# Final primary cohort
# 23 subjects / 38 recordings
# ---------------------------------------------------------

final_subjects = {
    "NIH1", "NIH2", "NIH3", "NIH4", "NIH5",
    "NIH6", "NIH7", "NIH8", "NIH9", "NIH10", "NIH11",
    "PY18N002", "PY18N013",
    "jh103", "jh105",
    "pt1", "pt2", "pt3",
    "umf001", "umf002", "umf003", "umf004", "umf005"
}

# Find all recordings
all_edf_files = sorted(data_dir.rglob("*_ieeg.edf"))

print("All recordings found:", len(all_edf_files))

# Keep only recordings belonging to final cohort
edf_files = []

for f in all_edf_files:
    match = re.search(r"sub-([^_]+)", f.name)

    if match is None:
        continue

    subject = match.group(1)

    if subject in final_subjects:
        edf_files.append(f)

print("Final cohort recordings:", len(edf_files))

# Show exactly what was selected
for f in edf_files:
    print(f.name)

All recordings found: 54
Final cohort recordings: 38
sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH2_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH3_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH4_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH5_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH6_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH7_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH8_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-NIH9_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-PY18N002_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-PY18N013_ses-extraoperative_task-interictal_run-01_ieeg.edf
sub-jh103_ses-extraoperative_task-interictalasleep_run-01_ieeg.edf
sub-jh103_ses-extraoperative_task-interictalawake_run-01_ieeg.edf
sub-jh105_ses-extraopera

## 2. Bad channels + 60 Hz notch

In [4]:
import re
import pandas as pd
import mne
from pathlib import Path


# ============================================================
# OUTPUT
# ============================================================

processed_dir = Path("/workspace/ds003876_processed")
notch_dir = processed_dir / "notch"
notch_dir.mkdir(parents=True, exist_ok=True)

notch_files = []


# ============================================================
# FINAL CHANNEL NORMALIZATION
# ============================================================

def normalize_channel_name(name):

    x = str(name).strip().upper()

    # Remove leading EEG
    x = re.sub(r"^EEG\s+", "", x)

    # Remove POL prefix
    x = re.sub(r"^POL\s+", "", x)

    # IMPORTANT:
    # DO NOT remove '$'
    #
    # $RPTO1 must remain different from RPTO1

    # Remove EDF reference suffixes
    #
    # RATO5-Re  -> RATO5
    # RAD5-Ref  -> RAD5
    # RSPI14-R  -> RSPI14
    x = re.sub(r"-(?:REF|RE|R)$", "", x)

    # Remove bipolar suffix
    #
    # RAI12-G2 -> RAI12
    x = re.sub(r"-G\d+$", "", x)

    # Special jh105 duplicate names
    #
    # EDF: POL-0 / POL-1
    # TSV: -0 / -1
    if x == "POL-0":
        return "-0"

    if x == "POL-1":
        return "-1"

    return x.strip()


# ============================================================
# PROCESS ALL 38 RECORDINGS
# ============================================================

for edf in edf_files:

    print(f"\nProcessing: {edf.name}")

    # --------------------------------------------------------
    # Load EDF
    # --------------------------------------------------------

    raw = mne.io.read_raw_edf(
        edf,
        preload=True,
        verbose=False
    )

    before = len(raw.ch_names)

    # --------------------------------------------------------
    # Matching TSV
    # --------------------------------------------------------

    tsv = edf.with_name(
        edf.name.replace(
            "_ieeg.edf",
            "_channels.tsv"
        )
    )

    if not tsv.exists():
        raise FileNotFoundError(
            f"Missing channels.tsv:\n{tsv}"
        )

    ch_df = pd.read_csv(
        tsv,
        sep="\t"
    )

    # --------------------------------------------------------
    # Get named bad channels
    # --------------------------------------------------------

    bad_rows = ch_df[
        ch_df["status"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("bad")
    ]

    # Ignore NaN channel names
    bad_rows = bad_rows[
        bad_rows["name"].notna()
    ]

    bad_names = set(
        bad_rows["name"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # --------------------------------------------------------
    # EDF -> normalized name
    # --------------------------------------------------------

    bad_edf_channels = []

    for edf_name in raw.ch_names:

        normalized = normalize_channel_name(
            edf_name
        )

        if normalized in bad_names:
            bad_edf_channels.append(edf_name)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    expected_bad = len(bad_names)
    matched_bad = len(bad_edf_channels)

    if expected_bad != matched_bad:

        raise RuntimeError(
            f"\nChannel matching failed:\n"
            f"{edf.name}\n"
            f"TSV bad channels = {expected_bad}\n"
            f"Matched EDF channels = {matched_bad}"
        )

    # --------------------------------------------------------
    # Remove bad channels
    # --------------------------------------------------------

    raw.drop_channels(
        bad_edf_channels,
        on_missing="ignore"
    )

    after = len(raw.ch_names)

    print(
        f"{before} -> {after} "
        f"({before - after} removed)"
    )

    # --------------------------------------------------------
    # 60 Hz notch
    # --------------------------------------------------------

    raw.notch_filter(
        freqs=[60],
        verbose=False
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    out = notch_dir / f"{edf.stem}.fif"

    raw.save(
        out,
        overwrite=True,
        verbose=False
    )

    notch_files.append(out)

    del raw


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 60)
print("BAD CHANNEL REMOVAL + 60 Hz NOTCH COMPLETE")
print("=" * 60)
print("Notch files:", len(notch_files))
print("Expected:", 38)


Processing: sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg.edf
186 -> 168 (18 removed)

Processing: sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg.edf
114 -> 82 (32 removed)

Processing: sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg.edf
121 -> 112 (9 removed)

Processing: sub-NIH2_ses-extraoperative_task-interictal_run-01_ieeg.edf
111 -> 90 (21 removed)

Processing: sub-NIH3_ses-extraoperative_task-interictal_run-01_ieeg.edf
134 -> 87 (47 removed)

Processing: sub-NIH4_ses-extraoperative_task-interictal_run-01_ieeg.edf
107 -> 94 (13 removed)

Processing: sub-NIH5_ses-extraoperative_task-interictal_run-01_ieeg.edf
118 -> 97 (21 removed)

Processing: sub-NIH6_ses-extraoperative_task-interictal_run-01_ieeg.edf
193 -> 185 (8 removed)

Processing: sub-NIH7_ses-extraoperative_task-interictal_run-01_ieeg.edf
146 -> 105 (41 removed)

Processing: sub-NIH8_ses-extraoperative_task-interictal_run-01_ieeg.edf
170 -> 153 (17 removed)

Processing: sub-NIH9_ses-extraoper

/tmp/ipykernel_1097/173796777.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'POL'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(


101 -> 85 (16 removed)

Processing: sub-jh105_ses-extraoperative_task-interictalawake_run-01_ieeg.edf


/tmp/ipykernel_1097/173796777.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'POL'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(


101 -> 85 (16 removed)

Processing: sub-pt1_ses-extraoperative_task-interictalasleep_run-01_ieeg.edf
98 -> 85 (13 removed)

Processing: sub-pt1_ses-extraoperative_task-interictalasleep_run-02_ieeg.edf
98 -> 85 (13 removed)

Processing: sub-pt1_ses-extraoperative_task-interictalawake_run-01_ieeg.edf
98 -> 85 (13 removed)

Processing: sub-pt1_ses-extraoperative_task-interictalawake_run-02_ieeg.edf
98 -> 85 (13 removed)

Processing: sub-pt2_ses-extraoperative_task-interictalasleep_run-01_ieeg.edf
86 -> 62 (24 removed)

Processing: sub-pt2_ses-extraoperative_task-interictalasleep_run-02_ieeg.edf
86 -> 62 (24 removed)

Processing: sub-pt2_ses-extraoperative_task-interictalawake_run-01_ieeg.edf
86 -> 62 (24 removed)

Processing: sub-pt2_ses-extraoperative_task-interictalawake_run-02_ieeg.edf
86 -> 62 (24 removed)

Processing: sub-pt3_ses-extraoperative_task-interictalasleep_run-01_ieeg.edf
135 -> 97 (38 removed)

Processing: sub-pt3_ses-extraoperative_task-interictalasleep_run-02_ieeg.edf
13

## 3. Common-average reference

In [5]:
car_dir = processed_dir / "car"
car_dir.mkdir(parents=True, exist_ok=True)

car_files = []

for i, f in enumerate(notch_files, start=1):

    print(f"[{i}/{len(notch_files)}] Processing {f.name}", flush=True)

    raw = mne.io.read_raw_fif(
        f,
        preload=True,
        verbose=False
    )

    raw.set_eeg_reference(
        "average",
        projection=False,
        verbose=False
    )

    out = car_dir / f.name

    raw.save(
        out,
        overwrite=True,
        verbose=False
    )

    car_files.append(out)

    del raw

    print(f"    saved: {out.name}", flush=True)

print("\nCAR files:", len(car_files))

[1/38] Processing sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg.fif
[2/38] Processing sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg.fif
[3/38] Processing sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg.fif
[4/38] Processing sub-NIH2_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH2_ses-extraoperative_task-interictal_run-01_ieeg.fif
[5/38] Processing sub-NIH3_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH3_ses-extraoperative_task-interictal_run-01_ieeg.fif
[6/38] Processing sub-NIH4_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH4_ses-extraoperative_task-interictal_run-01_ieeg.fif
[7/38] Processing sub-NIH5_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH5_ses-

## 4. Resample to 500 Hz

In [6]:
resampled_dir = processed_dir / "resampled_500"
resampled_dir.mkdir(parents=True, exist_ok=True)

resampled_files = []

for i, f in enumerate(car_files, start=1):

    print(f"[{i}/{len(car_files)}] Resampling {f.name}", flush=True)

    raw = mne.io.read_raw_fif(
        f,
        preload=True,
        verbose=False
    )

    raw.resample(
        500,
        npad="auto",
        verbose=False
    )

    out = resampled_dir / f.name

    raw.save(
        out,
        overwrite=True,
        verbose=False
    )

    resampled_files.append(out)

    print(
        f"    {raw.info['sfreq']:.1f} Hz",
        flush=True
    )

    del raw

print("\nResampled files:", len(resampled_files))

[1/38] Resampling sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[2/38] Resampling sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[3/38] Resampling sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[4/38] Resampling sub-NIH2_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[5/38] Resampling sub-NIH3_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[6/38] Resampling sub-NIH4_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[7/38] Resampling sub-NIH5_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[8/38] Resampling sub-NIH6_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[9/38] Resampling sub-NIH7_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[10/38] Resampling sub-NIH8_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.0 Hz
[11/38] Resampling sub-NIH9_ses-extraoperative_task-interictal_run-01_ieeg.fif
    500.

## 5. 0.5–150 Hz bandpass

In [7]:
bandpass_dir = processed_dir / "bandpass_500"
bandpass_dir.mkdir(parents=True, exist_ok=True)

bandpass_files = []

for i, f in enumerate(resampled_files, start=1):

    print(f"[{i}/{len(resampled_files)}] Band-pass filtering {f.name}", flush=True)

    raw = mne.io.read_raw_fif(
        f,
        preload=True,
        verbose=False
    )

    raw.filter(
        l_freq=0.5,
        h_freq=150.0,
        verbose=False
    )

    out = bandpass_dir / f.name

    raw.save(
        out,
        overwrite=True,
        verbose=False
    )

    bandpass_files.append(out)

    print(
        f"    saved: {out.name} | "
        f"sfreq={raw.info['sfreq']:.1f} Hz | "
        f"channels={len(raw.ch_names)}",
        flush=True
    )

    del raw

print("\nBand-pass files:", len(bandpass_files))

[1/38] Band-pass filtering sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg.fif | sfreq=500.0 Hz | channels=168
[2/38] Band-pass filtering sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg.fif | sfreq=500.0 Hz | channels=82
[3/38] Band-pass filtering sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg.fif | sfreq=500.0 Hz | channels=112
[4/38] Band-pass filtering sub-NIH2_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH2_ses-extraoperative_task-interictal_run-01_ieeg.fif | sfreq=500.0 Hz | channels=90
[5/38] Band-pass filtering sub-NIH3_ses-extraoperative_task-interictal_run-01_ieeg.fif
    saved: sub-NIH3_ses-extraoperative_task-interictal_run-01_ieeg.fif | sfreq=500.0 Hz | channels=87
[6/38] Band-pass filtering sub-NIH4_ses-extraoper

## 6. 10-second non-overlapping windows

In [8]:
windows_dir = processed_dir / "windows_10s"
windows_dir.mkdir(parents=True, exist_ok=True)

window_metadata = []

for i, f in enumerate(bandpass_files, start=1):

    print(f"[{i}/{len(bandpass_files)}] Windowing {f.name}", flush=True)

    raw = mne.io.read_raw_fif(
        f,
        preload=True,
        verbose=False
    )

    sfreq = raw.info["sfreq"]
    window_samples = int(10 * sfreq)

    n_samples = raw.n_times
    n_windows = n_samples // window_samples

    # Keep only complete 10-second windows
    for w in range(n_windows):

        start_sample = w * window_samples
        stop_sample = start_sample + window_samples

        data = raw.get_data(
            start=start_sample,
            stop=stop_sample
        )

        start_sec = start_sample / sfreq
        end_sec = stop_sample / sfreq

        window_metadata.append({
            "subject": f.name.split("_")[0],
            "recording": f.name,
            "window_id": w,
            "start_sec": start_sec,
            "end_sec": end_sec,
            "channels": data.shape[0],
            "samples": data.shape[1],
            "sampling_frequency": sfreq
        })

        # Save window
        out = windows_dir / (
            f"{f.stem}_window-{w:04d}.npy"
        )

        np.save(out, data)

    print(
        f"    {n_windows} windows | "
        f"{window_samples} samples/window",
        flush=True
    )

    del raw


window_metadata = pd.DataFrame(window_metadata)

metadata_file = windows_dir / "window_metadata.csv"
window_metadata.to_csv(
    metadata_file,
    index=False
)

print("\nTotal windows:", len(window_metadata))
print("Metadata:", metadata_file)

[1/38] Windowing sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg.fif
    57 windows | 5000 samples/window
[2/38] Windowing sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg.fif
    60 windows | 5000 samples/window
[3/38] Windowing sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg.fif
    60 windows | 5000 samples/window
[4/38] Windowing sub-NIH2_ses-extraoperative_task-interictal_run-01_ieeg.fif
    60 windows | 5000 samples/window
[5/38] Windowing sub-NIH3_ses-extraoperative_task-interictal_run-01_ieeg.fif
    60 windows | 5000 samples/window
[6/38] Windowing sub-NIH4_ses-extraoperative_task-interictal_run-01_ieeg.fif
    60 windows | 5000 samples/window
[7/38] Windowing sub-NIH5_ses-extraoperative_task-interictal_run-01_ieeg.fif
    60 windows | 5000 samples/window
[8/38] Windowing sub-NIH6_ses-extraoperative_task-interictal_run-01_ieeg.fif
    60 windows | 5000 samples/window
[9/38] Windowing sub-NIH7_ses-extraoperative_task-interictal_run-01_ieeg.fif
    60 wi

## 8. Spectral features

**Proposal frequency bands:** delta 1–4 Hz, theta 4–8 Hz, alpha 8–13 Hz, beta 13–30 Hz, low gamma 30–80 Hz, and high gamma 80–150 Hz.

In [11]:
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy.stats import kurtosis
from pathlib import Path

spectral_dir = processed_dir / "spectral_features"
spectral_dir.mkdir(parents=True, exist_ok=True)

bands = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "low_gamma": (30, 80),
    "high_gamma": (80, 150)
}

all_features = []

window_files = sorted(windows_dir.glob("*.npy"))

print("Window files:", len(window_files))

for i, f in enumerate(window_files, start=1):

    data = np.load(f)

    # data shape = channels × samples
    n_channels, n_samples = data.shape

    sfreq = 500

    # ---------------------------------------------------------
    # Parse recording / window information from filename
    # ---------------------------------------------------------

    window_name = f.stem

    parts = window_name.split("_window-")
    recording_name = parts[0]
    window_id = int(parts[1])

    subject = recording_name.split("_")[0]

    # ---------------------------------------------------------
    # Compute Welch PSD for all channels
    # ---------------------------------------------------------

    freqs, psd = welch(
        data,
        fs=sfreq,
        nperseg=min(1024, n_samples),
        axis=1
    )

    # ---------------------------------------------------------
    # Features per channel
    # ---------------------------------------------------------

    for ch_idx in range(n_channels):

        x = data[ch_idx]
        p = psd[ch_idx]

        row = {
            "subject": subject,
            "recording": recording_name,
            "window_id": window_id,
            "channel_index": ch_idx
        }

        # -----------------------------------------------------
        # Band powers
        # -----------------------------------------------------

        for band_name, (low, high) in bands.items():

            mask = (freqs >= low) & (freqs < high)

            row[f"{band_name}_power"] = np.trapz(
                p[mask],
                freqs[mask]
            )

        # -----------------------------------------------------
        # Spectral entropy
        # -----------------------------------------------------

        entropy_mask = (
            (freqs >= 1) &
            (freqs <= 150)
        )

        p_entropy = p[entropy_mask]

        total_power = np.sum(p_entropy)

        if total_power > 0:
            p_norm = p_entropy / total_power
            spectral_entropy = -np.sum(
                p_norm * np.log2(
                    p_norm + 1e-12
                )
            )
        else:
            spectral_entropy = np.nan

        row["spectral_entropy"] = spectral_entropy

        # -----------------------------------------------------
        # Line length
        # -----------------------------------------------------

        row["line_length"] = np.sum(
            np.abs(np.diff(x))
        )

        # -----------------------------------------------------
        # Variance
        # -----------------------------------------------------

        row["variance"] = np.var(x)

        # -----------------------------------------------------
        # Kurtosis
        # -----------------------------------------------------

        row["kurtosis"] = kurtosis(
            x,
            fisher=True,
            bias=False
        )

        all_features.append(row)

    if i % 100 == 0 or i == len(window_files):
        print(
            f"[{i}/{len(window_files)}] "
            f"windows processed",
            flush=True
        )


spectral_features = pd.DataFrame(
    all_features
)

out_file = spectral_dir / "spectral_features.csv"

spectral_features.to_csv(
    out_file,
    index=False
)

print("\nDone.")
print("Rows:", len(spectral_features))
print("Columns:", len(spectral_features.columns))
print("Saved:", out_file)

Window files: 1401
[100/1401] windows processed
[200/1401] windows processed
[300/1401] windows processed
[400/1401] windows processed
[500/1401] windows processed
[600/1401] windows processed
[700/1401] windows processed
[800/1401] windows processed
[900/1401] windows processed
[1000/1401] windows processed
[1100/1401] windows processed
[1200/1401] windows processed
[1300/1401] windows processed
[1400/1401] windows processed
[1401/1401] windows processed

Done.
Rows: 141121
Columns: 14
Saved: /workspace/ds003876_processed/spectral_features/spectral_features.csv


## 9. Aperiodic 1/f exponent and offset

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd

from scipy.signal import welch
from joblib import Parallel, delayed
from specparam import SpectralModel


# ============================================================
# OUTPUT
# ============================================================

aperiodic_dir = processed_dir / "aperiodic_features"
aperiodic_dir.mkdir(parents=True, exist_ok=True)

window_files = sorted(windows_dir.glob("*.npy"))

print("Window files:", len(window_files))


# ============================================================
# SETTINGS
# ============================================================

SFREQ = 500
FIT_RANGE = [1, 150]
N_JOBS = 64


# ============================================================
# PROCESS ONE WINDOW
# ============================================================

def process_window(f):

    data = np.load(f)

    n_channels, n_samples = data.shape

    parts = f.stem.split("_window-")

    recording_name = parts[0]
    window_id = int(parts[1])
    subject = recording_name.split("_")[0]

    # --------------------------------------------------------
    # PSD
    # --------------------------------------------------------

    freqs, psd = welch(
        data,
        fs=SFREQ,
        nperseg=min(1024, n_samples),
        axis=1
    )

    freq_mask = (
        (freqs >= FIT_RANGE[0]) &
        (freqs <= FIT_RANGE[1])
    )

    fit_freqs = freqs[freq_mask]

    rows = []

    # --------------------------------------------------------
    # Fit each channel
    # --------------------------------------------------------

    for ch_idx in range(n_channels):

        power = psd[ch_idx, freq_mask]

        try:

            model = SpectralModel(
                aperiodic_mode="fixed",
                verbose=False
            )

            model.fit(
                fit_freqs,
                power
            )

            # Correct API for our specparam version
            aperiodic_params = (
                model.results.params.aperiodic.params
            )

            # Fixed mode:
            # [offset, exponent]

            offset = float(
                aperiodic_params[0]
            )

            exponent = float(
                aperiodic_params[1]
            )

        except Exception:

            offset = np.nan
            exponent = np.nan

        rows.append({
            "subject": subject,
            "recording": recording_name,
            "window_id": window_id,
            "channel_index": ch_idx,
            "aperiodic_offset": offset,
            "aperiodic_exponent": exponent
        })

    return rows


# ============================================================
# PARALLEL PROCESSING
# ============================================================

results = Parallel(
    n_jobs=N_JOBS,
    backend="loky",
    verbose=10
)(
    delayed(process_window)(f)
    for f in window_files
)


# ============================================================
# COMBINE
# ============================================================

aperiodic_rows = []

for result in results:
    aperiodic_rows.extend(result)

aperiodic_features = pd.DataFrame(
    aperiodic_rows
)


# ============================================================
# SAVE
# ============================================================

out_file = (
    aperiodic_dir /
    "aperiodic_features.csv"
)

aperiodic_features.to_csv(
    out_file,
    index=False
)


# ============================================================
# VALIDATION
# ============================================================

print("\nDone.")
print("Rows:", len(aperiodic_features))
print("Columns:", len(aperiodic_features.columns))
print("Saved:", out_file)

print(
    "\nMissing exponent:",
    aperiodic_features[
        "aperiodic_exponent"
    ].isna().sum()
)

print(
    "Missing offset:",
    aperiodic_features[
        "aperiodic_offset"
    ].isna().sum()
)

print(
    "\nExample results:"
)

print(
    aperiodic_features[
        [
            "aperiodic_offset",
            "aperiodic_exponent"
        ]
    ].head()
)

Window files: 1401


[Parallel(n_jobs=64)]: Using backend LokyBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done  17 tasks      | elapsed:   10.0s
[Parallel(n_jobs=64)]: Done  34 tasks      | elapsed:   18.4s
[Parallel(n_jobs=64)]: Done  53 tasks      | elapsed:   23.1s
[Parallel(n_jobs=64)]: Done  72 tasks      | elapsed:   29.5s
[Parallel(n_jobs=64)]: Done  93 tasks      | elapsed:   36.6s
[Parallel(n_jobs=64)]: Done 114 tasks      | elapsed:   41.0s
[Parallel(n_jobs=64)]: Done 137 tasks      | elapsed:   46.3s
[Parallel(n_jobs=64)]: Done 160 tasks      | elapsed:   53.1s
[Parallel(n_jobs=64)]: Done 185 tasks      | elapsed:  1.5min
[Parallel(n_jobs=64)]: Done 210 tasks      | elapsed:  1.6min
[Parallel(n_jobs=64)]: Done 237 tasks      | elapsed:  1.9min
[Parallel(n_jobs=64)]: Done 264 tasks      | elapsed:  2.2min
[Parallel(n_jobs=64)]: Done 293 tasks      | elapsed:  2.4min
[Parallel(n_jobs=64)]: Done 322 tasks      | elapsed:  3.1min
[Parallel(n_jobs=64)]: Done 353 tasks      | elapsed:  


Done.
Rows: 141121
Columns: 6
Saved: /workspace/ds003876_processed/aperiodic_features/aperiodic_features.csv

Missing exponent: 0
Missing offset: 0

Example results:
   aperiodic_offset  aperiodic_exponent
0         -7.964027            2.085052
1         -8.046055            2.202870
2         -7.819328            2.302424
3         -7.903249            2.366700
4         -8.192882            2.265265


## 10. Parallel coherence connectivity

In [18]:
import numpy as np
from pathlib import Path
from scipy.signal import coherence
from joblib import Parallel, delayed


# ============================================================
# OUTPUT
# ============================================================

coherence_dir = (
    processed_dir /
    "connectivity" /
    "coherence"
)

coherence_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# SETTINGS
# ============================================================

SFREQ = 500

bands = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "low_gamma": (30, 80),
    "high_gamma": (80, 150)
}

band_names = list(bands.keys())

N_JOBS = 64


# ============================================================
# GET WINDOW FILES
# ============================================================

window_files = sorted(
    windows_dir.glob("*.npy")
)

print("Total window files:", len(window_files))


# ============================================================
# GROUP WINDOWS BY RECORDING
# ============================================================

recording_windows = {}

for f in window_files:

    parts = f.stem.split("_window-")

    recording_name = parts[0]

    recording_windows.setdefault(
        recording_name,
        []
    ).append(f)

print(
    "Recordings:",
    len(recording_windows)
)


# ============================================================
# COHERENCE FOR ONE CHANNEL PAIR
# ============================================================

def compute_pair_coherence(
    data,
    ch1,
    ch2
):

    x = data[ch1]
    y = data[ch2]

    freqs, coh = coherence(
        x,
        y,
        fs=SFREQ,
        nperseg=1024
    )

    band_values = []

    for low, high in bands.values():

        mask = (
            (freqs >= low) &
            (freqs < high)
        )

        if np.any(mask):
            value = np.mean(
                coh[mask]
            )
        else:
            value = np.nan

        band_values.append(value)

    return ch1, ch2, band_values


# ============================================================
# PROCESS ONE RECORDING
# ============================================================

for recording_idx, (
    recording_name,
    files
) in enumerate(
    sorted(recording_windows.items()),
    start=1
):

    print(
        f"\n[{recording_idx}/{len(recording_windows)}] "
        f"{recording_name}",
        flush=True
    )

    # --------------------------------------------------------
    # Sort windows by window number
    # --------------------------------------------------------

    files = sorted(
        files,
        key=lambda f: int(
            f.stem.split("_window-")[1]
        )
    )

    # --------------------------------------------------------
    # Load windows
    # --------------------------------------------------------

    first_data = np.load(
        files[0]
    )

    n_channels = first_data.shape[0]
    n_windows = len(files)

    print(
        f"    Windows: {n_windows}",
        flush=True
    )

    print(
        f"    Channels: {n_channels}",
        flush=True
    )

    # --------------------------------------------------------
    # Allocate output
    #
    # windows × channels × channels × bands
    # --------------------------------------------------------

    connectivity = np.zeros(
        (
            n_windows,
            n_channels,
            n_channels,
            len(bands)
        ),
        dtype=np.float32
    )

    # --------------------------------------------------------
    # Process each window
    # --------------------------------------------------------

    for w, f in enumerate(files):

        data = np.load(f)

        # ----------------------------------------------------
        # Channel pairs
        # ----------------------------------------------------

        pairs = [
            (i, j)
            for i in range(n_channels)
            for j in range(i, n_channels)
        ]

        # ----------------------------------------------------
        # Parallel coherence
        # ----------------------------------------------------

        results = Parallel(
            n_jobs=N_JOBS,
            backend="loky"
        )(
            delayed(compute_pair_coherence)(
                data,
                i,
                j
            )
            for i, j in pairs
        )

        # ----------------------------------------------------
        # Fill symmetric matrix
        # ----------------------------------------------------

        for i, j, values in results:

            connectivity[
                w,
                i,
                j,
                :
            ] = values

            connectivity[
                w,
                j,
                i,
                :
            ] = values

        # ----------------------------------------------------
        # Diagonal = 1
        # ----------------------------------------------------

        for b in range(
            len(bands)
        ):
            np.fill_diagonal(
                connectivity[
                    w, :, :, b
                ],
                1.0
            )

        if (
            (w + 1) % 10 == 0
            or w == n_windows - 1
        ):
            print(
                f"    Window {w + 1}/{n_windows}",
                flush=True
            )

        del data

    # --------------------------------------------------------
    # Save recording-level connectivity
    # --------------------------------------------------------

    out = (
        coherence_dir /
        f"{recording_name}_coherence.npy"
    )

    np.save(
        out,
        connectivity
    )

    print(
        f"    Saved: {out.name}",
        flush=True
    )

    print(
        f"    Shape: {connectivity.shape}",
        flush=True
    )

    del connectivity


print("\n" + "=" * 70)
print("COHERENCE CONNECTIVITY COMPLETE")
print("=" * 70)
print(
    "Output directory:",
    coherence_dir
)

Total window files: 1401
Recordings: 38

[1/38] sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg
    Windows: 60
    Channels: 82
    Window 10/60
    Window 20/60
    Window 30/60
    Window 40/60
    Window 50/60
    Window 60/60
    Saved: sub-NIH10_ses-extraoperative_task-interictal_run-01_ieeg_coherence.npy
    Shape: (60, 82, 82, 6)

[2/38] sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg
    Windows: 60
    Channels: 112
    Window 10/60
    Window 20/60
    Window 30/60
    Window 40/60
    Window 50/60
    Window 60/60
    Saved: sub-NIH11_ses-extraoperative_task-interictal_run-01_ieeg_coherence.npy
    Shape: (60, 112, 112, 6)

[3/38] sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg
    Windows: 57
    Channels: 168
    Window 10/57
    Window 20/57
    Window 30/57
    Window 40/57
    Window 50/57
    Window 57/57
    Saved: sub-NIH1_ses-extraoperative_task-interictal_run-01_ieeg_coherence.npy
    Shape: (57, 168, 168, 6)

[4/38] sub-NIH2_ses-extraope